# Primitives

Primitive is the smallest processing instruction from which one can create something useful for a given abstraction level. As we move towards larger QPUs, our focus shifts from interacting with individual qubit signals to viewing quantum devices as systems that perform necessary tasks.

The two most common tasks for quantum computers are sampling quantum state and calculating expectation values. These tasks motivated the design of the Qiskit primitives: 

`Estimator` computes expectation values of observables with respect to states prepared by quantum circuits.

`Sampler` samples the output register from quantum circuit execution.

The computational model introduced by the Qiskit primitives moves quantum programming closer to where classical programming is today, where the focus is less on the hardware details and more on the results you are trying to achieve.

# Primitive Definitions and Implementations

There are two types of Qiskit primitives:

* Base Classes - The base primitives are abstract classes that define a common interface for implementing primitives. One should use these if they are interested in creating their own primitve-based execution model for a specific provider. These classes might also be useful for those who want to do highly customized processing and find that the existing primitives implementations are too simple for their needs. General users will not directly use the base classes.

`BaseEstimatorV1` and `BaseSamplerV1` - Base class for estimator and sampler respectively, all implementations inherit from this class.

`BaseEstimatorV2` and `BaseSamplerV2` - The Qiskit reference primitives follow these interface specifications.


* Implementations - All primitives are created from the base classes; therefore, they have the same general structure and usage. For example, the format of the input for all Estimator primitives is the same. However, there are differences in implementations that make them unique.

The Qiskit Runtime primitives, `EstimatorV2` and `SamplerV2`, provide a more sophisticated implementation (for example, by including error mitigation) as a cloud-based service. This implementation of the base primitives is used to access IBM Quantum® hardware. They are accessed through Qiskit Runtime.

`StatevectorEstimator` and `StatevectorSampler` - Reference implementations of the primitives that use the simulator built into Qiskit. They are built with the Qiskit quantum_info module, producing results based on ideal statevector simulations. They are accessed through Qiskit.

`BackendEstimatorV2` and `BackendSamplerV2` - You can use these classes to “wrap” any quantum computing resource into a primitive. This lets you write primitive-style code for providers that don’t yet have a primitives-based interface. These classes can be used just like the regular Sampler and Estimator, except they should be initialized with an additional backend argument for selecting which quantum computer to run on. They are accessed by using Qiskit.


# Benefits of Qiskit Primitives

* Qiskit users can write quantum code for a specific QPU without having to explicitly manage every detail.

* Because of the additional layer of abstraction, you might be able to more easily access advanced hardware capabilities of a given provider.
Eg: With Qiskit Runtime primitives, you can take advantage of the latest advancements in error mitigation and suppression by toggling options such as the primitive's resilience_level, rather than building your own implementation of these techniques.

* For hardware providers, implementing primitives natively means you can provide your users with a more “out-of-the-box” way to access your hardware features such as advanced post-processing techniques. It is therefore easier for your users to benefit from your hardware's best capabilities.


# Primitive Inputs & Outputs

With these primitives you can use a data structure known as a Primitive Unified Bloc (PUB) to efficiently define vectorized workloads. These PUBs are the fundamental unit of work for workload execution. They are used as inputs to the run() method for the Sampler and Estimator primitives, which execute the defined workload as a job. Then, after the job has completed, the results are returned in a format that depends on the PUBs used and any specified options.

When invoking a primitive's run() method, the main argument that is required is a list of one or more tuples -- one for each circuit being executed by the primitive. Each of these tuples is considered a PUB, and the required elements of each tuple in the list depends on the primitive used.

#### Estimator PUB

* A single QuantumCircuit, which may contain one or more Parameter objects
* A list of one or more observables, which specify the expectation values to estimate, arranged into an array (for example, a single observable represented as a 0-d array, a list of observables as a 1-d array, and so on). The data can be in any one of the ObservablesArrayLike format such as Pauli, SparsePauliOp, PauliList, or str.
* A collection of parameter values to bind the circuit against. This can be specified as a single array-like object where the last index is over circuit Parameter objects, or omitted (or equivalently, set to None) if the circuit has no Parameter objects.
* (Optionally) a target precision for expectation values to estimate

#### Sampler PUB


* A single QuantumCircuit, which may contain one or more Parameter objects Note: These circuits must also include measurement instructions for each of the qubits to be sampled.
* A collection of parameter values to bind the circuit against θk (only needed if any Parameter objects are used that must be bound at runtime)
*(Optionally) a number of shots to measure the circuit with

In [3]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
    ClassicalRegister,
    QuantumRegister,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives.containers import BitArray
from qiskit.primitives import StatevectorEstimator


import numpy as np

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

# Transpile the circuit without providing a backend
pm = generate_preset_pass_manager(optimization_level=1)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 10),
        np.linspace(-4 * np.pi, 4 * np.pi, 10),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator = StatevectorEstimator()
estimator_pub = (transpiled_circuit, observables, params)

# Run the transpiled circuit
# using the set of parameters and observables.

job = estimator.run([estimator_pub])
result = job.result()

#### Broadcasting rules
The PUBs aggregate elements from multiple arrays (observables and parameter values) by following the same broadcasting rules as NumPy. 

Rules:

* Input arrays do not need to have the same number of dimensions.
 * The resulting array will have the same number of dimensions as the input array with the largest dimension.
 * The size of each dimension is the largest size of the corresponding dimension.
 * Missing dimensions are assumed to have size one.
* Shape comparisons start with the rightmost dimension and continue to the left.
* Two dimensions are compatible if their sizes are equal or if one of them is 1.

Examples of array pairs that broadcast:

A1     (1d array):      1

A2     (2d array):  3 x 5

Result (2d array):  3 x 5



A1     (3d array):  11 x 2 x 7

A2     (3d array):  11 x 1 x 7

Result (3d array):  11 x 2 x 7


Examples of array pairs that do not broadcast:

A1     (1d array):  5

A2     (1d array):  3


A1     (2d array):      2 x 1

A2     (3d array):  6 x 5 x 4 This would work if the middle dimension were 2, but it is 5.

In [4]:
import numpy as np
# Broadcast single observable
parameter_values = np.random.uniform(size=(5,))  # shape (5,)
observables = SparsePauliOp("ZZZ")  # shape ()
# >> pub result has shape (5,)

# Zip
parameter_values = np.random.uniform(size=(5,))  # shape (5,)
observables = [
    SparsePauliOp(pauli) for pauli in ["III", "XXX", "YYY", "ZZZ", "XYZ"]
]  # shape (5,)
# >> pub result has shape (5,)

# Outer/Product
parameter_values = np.random.uniform(size=(1, 6))  # shape (1, 6)
observables = [
    [SparsePauliOp(pauli)] for pauli in ["III", "XXX", "YYY", "ZZZ"]
]  # shape (4, 1)
# >> pub result has shape (4, 6)

# Standard nd generalization
parameter_values = np.random.uniform(size=(3, 6))  # shape (3, 6)
observables = [
    [
        [SparsePauliOp(["XII"])],
        [SparsePauliOp(["IXI"])],
        [SparsePauliOp(["IIX"])],
    ],
    [
        [SparsePauliOp(["ZII"])],
        [SparsePauliOp(["IZI"])],
        [SparsePauliOp(["IIZ"])],
    ],
]  # shape (2, 3, 1)
# >> pub result has shape (2, 3, 6)

Each SparsePauliOp counts as a single element in this context, regardless of the number of Paulis contained in the SparsePauliOp. Thus, for the purpose of these broadcasting rules, all of the following elements have the same shape:


a = SparsePauliOp("Z") # shape ()  
b = SparsePauliOp("IIIIZXYIZ") # shape ()  
c = SparsePauliOp.from_list(["XX", "XY", "IZ"]) # shape ()  
The following lists of operators, while equivalent in terms of information contained, have different shapes:  


list1 = SparsePauliOp.from_list(["XX", "XY", "IZ"]) # shape ()  
list2 = [SparsePauliOp("XX"), SparsePauliOp("XY"), SparsePauliOp("IZ")] # shape (3, )  


#### Overview of primitive outputs
Once one or more PUBs are sent to a QPU for execution and a job successfully completes, the data is returned as a PrimitiveResult container object. The PrimitiveResult contains an iterable list of PubResult objects that contain the execution results for each PUB. For example, a job submitted with 20 PUBs will return a PrimitiveResult object that contains a list of 20 PubResults, one corresponding to each PUB.

Each of these PubResult objects possess both a data and an optional metadata attribute. The data attribute is a customized DataBin that contains the the expectation value estimates in case of the Estimator, or samples of the circuit output in the case of the Sampler.

The data attribute might also include other implementation-specific information such as standard deviations. The metadata attribute can contain additional implementation-specific information about the execution of the associated PUB.



#### Estimator output

As stated previously, the data returned in the PubResult for the Estimator primitive depends on the implementation. For example, it might contain an array of expectation values (PubResult.data.evs) and associated standard deviations (PubResult.data.stds).

The below code snippet describes the PrimitiveResult (and associated PubResult) format for the job created above.

In [5]:
print(
    f"The result of the submitted job had {len(result)} PUB and has a value:\n {result}\n"
)
print(
    f"The associated PubResult of this job has the following data bins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    "Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the\n\
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). \n"
)
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 10), dtype=float64>), stds=np.ndarray(<shape=(3, 10), dtype=float64>), shape=(3, 10)), metadata={'target_precision': 0.0, 'circuit_metadata': {}})], metadata={'version': 2})

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 10), dtype=float64>), stds=np.ndarray(<shape=(3, 10), dtype=float64>), shape=(3, 10))

And this DataBin has attributes: dict_keys(['evs', 'stds'])
Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). 

The expectation values measured from this PUB are: 
[[ 3.06161700e-16  4.52395120e-01  4.36594428e-01  2.16506351e-01
   6.33718361e-01 -6.33718361e-01 -2.16506351e-01 -4.36594428e-01
  -4.52395120e-01 -3.06161700e-16]
 [ 1.22464680

#### Sampler output

When a Sampler job is completed successfully, the returned PrimitiveResult object contains a list of SamplerPubResults, one per PUB. The data bins of these SamplerPubResult objects are dict-like objects that contain one BitArray per ClassicalRegister in the circuit.

The BitArray class is a container for ordered shot data. In more detail, it stores the sampled bitstrings as bytes inside a two-dimensional array. The left-most axis of this array runs over ordered shots, while the right-most axis runs over bytes.

As a first example, let us look at the following ten-qubit circuit:

In [6]:
from qiskit.primitives import StatevectorSampler

# generate a ten-qubit GHZ circuit
circuit = QuantumCircuit(10)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure_all()

# transpile the circuit
transpiled_circuit = pm.run(circuit)

sampler = StatevectorSampler()

# run the Sampler job and retrieve the results

job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains one BitArray
data = result[0].data
print(f"Databin: {data}\n")

# to access the BitArray, use the key "meas", which is the default name of
# the classical register when this is added by the `measure_all` method
array = data.meas
print(f"BitArray: {array}\n")
print(f"The shape of register `meas` is {data.meas.array.shape}.\n")
print(f"The bytes in register `alpha`, shot by shot:\n{data.meas.array}\n")

Databin: DataBin(meas=BitArray(<shape=(), num_shots=1024, num_bits=10>))

BitArray: BitArray(<shape=(), num_shots=1024, num_bits=10>)

The shape of register `meas` is (1024, 2).

The bytes in register `alpha`, shot by shot:
[[  3 255]
 [  3 255]
 [  3 255]
 ...
 [  3 255]
 [  0   0]
 [  0   0]]



It can sometimes be convenient to convert away from the bytes format in the BitArray to bitstrings. The get_count method returns a dictionary mapping bitstrings to the number of times that they occurred.

In [7]:
# optionally, convert away from the native BitArray format to a dictionary format
counts = data.meas.get_counts()
print(f"Counts: {counts}")

Counts: {'1111111111': 475, '0000000000': 549}


When a circuit contains more than one classical register, the results are stored in different BitArray objects. The following example modifies the previous snippet by splitting the classical register into two distinct registers:

In [8]:
# generate a ten-qubit GHZ circuit with two classical registers
circuit = QuantumCircuit(
    qreg := QuantumRegister(10),
    alpha := ClassicalRegister(1, "alpha"),
    beta := ClassicalRegister(9, "beta"),
)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure([0], alpha)
circuit.measure(range(1, 10), beta)

# transpile the circuit
transpiled_circuit = pm.run(circuit)

# run the Sampler job and retrieve the results

job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains two BitArrays, one per register, and can be accessed
# as attributes using the registers' names
data = result[0].data
print(f"BitArray for register 'alpha': {data.alpha}")
print(f"BitArray for register 'beta': {data.beta}")

BitArray for register 'alpha': BitArray(<shape=(), num_shots=1024, num_bits=1>)
BitArray for register 'beta': BitArray(<shape=(), num_shots=1024, num_bits=9>)


#### Leveraging BitArray objects for performant post-processing
Since arrays generally offer better performance compared to dictionaries, it is advisable to perform any post-processing directly on the BitArray objects rather than on dictionaries of counts. The BitArray class offers a range of methods to perform some common post-processing operations:

In [9]:
print(f"The shape of register `alpha` is {data.alpha.array.shape}.")
print(f"The bytes in register `alpha`, shot by shot:\n{data.alpha.array}\n")

print(f"The shape of register `beta` is {data.beta.array.shape}.")
print(f"The bytes in register `beta`, shot by shot:\n{data.beta.array}\n")

# post-select the bitstrings of `beta` based on having sampled "1" in `alpha`
mask = data.alpha.array == "0b1"
ps_beta = data.beta[mask[:, 0]]
print(f"The shape of `beta` after post-selection is {ps_beta.array.shape}.")
print(f"The bytes in `beta` after post-selection:\n{ps_beta.array}")

# get a slice of `beta` to retrieve the first three bits
beta_sl_bits = data.beta.slice_bits([0, 1, 2])
print(
    f"The shape of `beta` after bit-wise slicing is {beta_sl_bits.array.shape}."
)
print(f"The bytes in `beta` after bit-wise slicing:\n{beta_sl_bits.array}\n")

# get a slice of `beta` to retrieve the bytes of the first five shots
beta_sl_shots = data.beta.slice_shots([0, 1, 2, 3, 4])
print(
    f"The shape of `beta` after shot-wise slicing is {beta_sl_shots.array.shape}."
)
print(
    f"The bytes in `beta` after shot-wise slicing:\n{beta_sl_shots.array}\n"
)

# calculate the expectation value of diagonal operators on `beta`
ops = [SparsePauliOp("ZZZZZZZZZ"), SparsePauliOp("IIIIIIIIZ")]
exp_vals = data.beta.expectation_values(ops)
for o, e in zip(ops, exp_vals):
    print(f"Exp. val. for observable `{o}` is: {e}")

# concatenate the bitstrings in `alpha` and `beta` to "merge" the results of the two
# registers
merged_results = BitArray.concatenate_bits([data.alpha, data.beta])
print(f"\nThe shape of the merged results is {merged_results.array.shape}.")
print(f"The bytes of the merged results:\n{merged_results.array}\n")

The shape of register `alpha` is (1024, 1).
The bytes in register `alpha`, shot by shot:
[[0]
 [1]
 [0]
 ...
 [1]
 [0]
 [0]]

The shape of register `beta` is (1024, 2).
The bytes in register `beta`, shot by shot:
[[  0   0]
 [  1 255]
 [  0   0]
 ...
 [  1 255]
 [  0   0]
 [  0   0]]

The shape of `beta` after post-selection is (0, 2).
The bytes in `beta` after post-selection:
[]
The shape of `beta` after bit-wise slicing is (1024, 1).
The bytes in `beta` after bit-wise slicing:
[[0]
 [7]
 [0]
 ...
 [7]
 [0]
 [0]]

The shape of `beta` after shot-wise slicing is (5, 2).
The bytes in `beta` after shot-wise slicing:
[[  0   0]
 [  1 255]
 [  0   0]
 [  1 255]
 [  1 255]]

Exp. val. for observable `SparsePauliOp(['ZZZZZZZZZ'],
              coeffs=[1.+0.j])` is: 0.00390625
Exp. val. for observable `SparsePauliOp(['IIIIIIIIZ'],
              coeffs=[1.+0.j])` is: 0.00390625

The shape of the merged results is (1024, 2).
The bytes of the merged results:
[[  0   0]
 [  3 255]
 [  0   0]
 ...


#### Result metadata
In addition to the execution results, the PrimitiveResult and PubResult objects contain an optional metadata attribute about the job that was submitted. The metadata returned (if any) is implementation-specific.

In [10]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'version' : 2,

The metadata of the PubResult result is:
'shots' : 1024,
'circuit_metadata' : {},
